<a href="https://colab.research.google.com/github/Sarvardev035/AI-ml-302-sec2/blob/main/sarvarbek_Sharipboyev_2nd_lab_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Lab Report: Multi-Layer Perceptron (MLP) Implementation from Scratch
Objective
The goal of this lab is to build, train, and evaluate a 2-layer Multi-Layer Perceptron (MLP) from scratch using NumPy. We derive and code vectorized backpropagation, verify gradient shapes, solve the non-linearly separable XOR problem, explore activation variations (ReLU vs. Sigmoid), test cross-entropy loss, and encapsulate the pipeline into a modular OOP class.

Part 1: Backpropagation Derivation & Gradient Computation
In this section, we complete the backpropagation step for a 2-layer MLP using Mean Squared Error (MSE) loss. The architecture consists of:

Input X∈R
N×2


Hidden weights W
1
​
 ∈R
2×4
 , bias b
1
​
 ∈R
1×4


Hidden activation A
1
​
 =σ(Z
1
​
 )∈R
N×4


Output weights W
2
​
 ∈R
4×1
 , bias b
2
​
 ∈R
1×1


Output activation A
2
​
 =σ(Z
2
​
 )∈R
N×1


Vectorized Gradient Formulas
Output loss derivative:

∂A
2
​

∂L
​
 =
N
2
​
 (A
2
​
 −Y)
Output pre-activation gradient:

δ
2
​
 =
∂Z
2
​

∂L
​
 =
∂A
2
​

∂L
​
 ⊙A
2
​
 ⊙(1−A
2
​
 )
Output parameter gradients:

∂W
2
​

∂L
​
 =A
1
T
​
 δ
2
​
 ,
∂b
2
​

∂L
​
 =
i=1
∑
N
​
 δ
2,i
​

Hidden layer error propagation:

∂A
1
​

∂L
​
 =δ
2
​
 W
2
T
​

Hidden pre-activation gradient:

δ
1
​
 =
∂Z
1
​

∂L
​
 =
∂A
1
​

∂L
​
 ⊙A
1
​
 ⊙(1−A
1
​
 )
Hidden parameter gradients:

∂W
1
​

∂L
​
 =X
T
 δ
1
​
 ,
∂b
1
​

∂L
​
 =
i=1
∑
N
​
 δ
1,i
​

Python Implementation (Cell 5e3b4c7c)
Python
import numpy as np
import matplotlib.pyplot as plt

# XOR Data Setup
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float64)
Y_xor = np.array([[0], [1], [1], [0]], dtype=np.float64)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def init_params(n_inputs=2, n_hidden=4, seed=42):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0.0, 1.0, size=(n_inputs, n_hidden))
    b1 = np.zeros((1, n_hidden))
    W2 = rng.normal(0.0, 1.0, size=(n_hidden, 1))
    b2 = np.zeros((1, 1))
    return W1, b1, W2, b2

def forward(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    A1 = sigmoid(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)
    return A1, A2

# Setup variables
W1, b1, W2, b2 = init_params()
X, Y = X_xor, Y_xor
N = X.shape[0]

# Forward pass
A1, A2 = forward(X, W1, b1, W2, b2)

# --- Backward Pass ---
# Output Layer
dA2 = 2 * (A2 - Y) / N
dZ2 = dA2 * A2 * (1 - A2)
dW2 = A1.T @ dZ2
db2 = np.sum(dZ2, axis=0, keepdims=True)

# Hidden Layer
dA1 = dZ2 @ W2.T
dZ1 = dA1 * A1 * (1 - A1)
dW1 = X.T @ dZ1
db1 = np.sum(dZ1, axis=0, keepdims=True)

# Gradient Verification
print("--- Shape Verification ---")
for name, grad, param in [("dW2", dW2, W2), ("db2", db2, b2),
                          ("dW1", dW1, W1), ("db1", db1, b1)]:
    print(f"{name:3s} shape: {str(grad.shape):10s} | Target shape: {str(param.shape):10s} | Match: {grad.shape == param.shape}")
Part 2: XOR Training Loop & Decision Boundary
Since XOR is a classic non-linearly separable problem, a single-layer perceptron fails to solve it. We train our 2-layer network for 10,000 iterations using Gradient Descent with a learning rate of η=1.0.

Python
# Re-initialize parameters
W1, b1, W2, b2 = init_params(n_inputs=2, n_hidden=4, seed=42)
lr = 1.0
epochs = 10000
losses = []

def mse_loss(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)

for epoch in range(epochs):
    # Forward
    Z1 = X_xor @ W1 + b1
    A1 = sigmoid(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)

    # Loss tracking
    loss = mse_loss(A2, Y_xor)
    losses.append(loss)

    # Backward
    dA2 = 2 * (A2 - Y_xor) / N
    dZ2 = dA2 * A2 * (1 - A2)
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * A1 * (1 - A1)
    dW1 = X_xor.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    # Updates
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

print("\n--- Training Results ---")
print(f"Final Loss: {losses[-1]:.6f}")
print("Predictions vs Targets:")
for x, pred, target in zip(X_xor, A2, Y_xor):
    print(f"Input: {x} | Predicted Probability: {pred[0]:.4f} | Target: {target[0]}")

# Visualization: Loss Curve & Decision Boundary
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Plot Loss
axes[0].plot(losses, color='navy', lw=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("XOR Training Loss Curve")
axes[0].grid(True, linestyle='--', alpha=0.6)

# Plot Decision Boundary
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
_, grid_preds = forward(grid, W1, b1, W2, b2)
zz = grid_preds.reshape(xx.shape)

contour = axes[1].contourf(xx, yy, zz, levels=20, cmap="Spectral", alpha=0.8)
plt.colorbar(contour, ax=axes[1], label="Predicted Probability P(Y=1)")

for label, marker, color in [(0, 'o', 'red'), (1, 's', 'blue')]:
    mask = Y_xor[:, 0] == label
    axes[1].scatter(X_xor[mask, 0], X_xor[mask, 1], s=120, c=color, marker=marker,
                     edgecolors="k", label=f"Class {label}")

axes[1].set_title("Learned Decision Boundary")
axes[1].set_xlabel("x1")
axes[1].set_ylabel("x2")
axes[1].legend()
plt.tight_layout()
plt.show()
Part 3: Laboratory Exercises & Analysis
Exercise 1: Replacing Sigmoid with ReLU in Hidden Layer
Mathematical Formulation:

ReLU(z)=max(0,z)

Derivative:

ReLU
′
 (z)={
1
0
​

if z>0
if z≤0
​

Using ReLU in the hidden layer helps mitigate vanishing gradients. However, because its derivative can shut off for negative inputs (z≤0), proper parameter initialization and a moderate learning rate are required to avoid "dead neurons."

Python
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

# Initialization with He scaling for ReLU
rng = np.random.default_rng(1)
W1_relu = rng.normal(0.0, np.sqrt(2.0 / 2), size=(2, 4))
b1_relu = np.zeros((1, 4))
W2_relu = rng.normal(0.0, np.sqrt(2.0 / 4), size=(4, 1))
b2_relu = np.zeros((1, 1))

lr = 0.1
epochs = 5000
losses_relu = []

for epoch in range(epochs):
    # Forward
    Z1 = X_xor @ W1_relu + b1_relu
    A1 = relu(Z1)
    Z2 = A1 @ W2_relu + b2_relu
    A2 = sigmoid(Z2)

    loss = mse_loss(A2, Y_xor)
    losses_relu.append(loss)

    # Backward
    dA2 = 2 * (A2 - Y_xor) / N
    dZ2 = dA2 * A2 * (1 - A2)
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = dZ2 @ W2_relu.T
    dZ1 = dA1 * relu_derivative(Z1)  # Gradient evaluated at Z1
    dW1 = X_xor.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    # Gradient Update
    W2_relu -= lr * dW2
    b2_relu -= lr * db2
    W1_relu -= lr * dW1
    b1_relu -= lr * db1

print("--- Exercise 1: ReLU Hidden Activation Results ---")
print(f"Final Loss: {losses_relu[-1]:.6f}")
print("Predictions:\n", np.round(A2, 4))
Exercise 2: Binary Cross-Entropy (BCE) Loss Implementation
Mathematical Derivation:
Binary Cross-Entropy Loss is defined as:

L
BCE
​
 =−
N
1
​

i=1
∑
N
​
 [y
i
​
 log(
y
^
​

i
​
 )+(1−y
i
​
 )log(1−
y
^
​

i
​
 )]
Taking the derivative of L
BCE
​
  with respect to output activation
y
^
​
 =A
2
​
 :

∂A
2
​

∂L
BCE
​

​
 =
N
1
​
 (
A
2
​
 (1−A
2
​
 )
A
2
​
 −Y
​
 )
When combined with Sigmoid output activation (A
2
​
 =σ(Z
2
​
 )), the output activation derivative σ
′
 (Z
2
​
 )=A
2
​
 (1−A
2
​
 ) cancels out algebraically:

δ
2
​
 =
∂Z
2
​

∂L
BCE
​

​
 =
∂A
2
​

∂L
BCE
​

​
 ⋅σ
′
 (Z
2
​
 )=
N
1
​
 (A
2
​
 −Y)
This simplifies the backward pass and significantly speeds up training convergence.

Python
def bce_loss(A2, Y, eps=1e-15):
    # Prevent log(0) numerical instability by clipping values
    A2_clipped = np.clip(A2, eps, 1 - eps)
    return -np.mean(Y * np.log(A2_clipped) + (1 - Y) * np.log(1 - A2_clipped))

# Training setup with BCE
W1, b1, W2, b2 = init_params(n_inputs=2, n_hidden=4, seed=42)
lr = 0.5
epochs = 2000
losses_bce = []

for epoch in range(epochs):
    # Forward
    Z1 = X_xor @ W1 + b1
    A1 = sigmoid(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)

    loss = bce_loss(A2, Y_xor)
    losses_bce.append(loss)

    # Backward: Combined BCE + Sigmoid simplification
    dZ2 = (A2 - Y_xor) / N
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * A1 * (1 - A1)
    dW1 = X_xor.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    # Updates
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

print("--- Exercise 2: Binary Cross-Entropy Results ---")
print(f"Final BCE Loss after 2000 Epochs: {losses_bce[-1]:.6f}")
print("Predictions:\n", np.round(A2, 4))
Exercise 3: Full Modular MLP Class Implementation
Below is a complete, reusable, vectorized Object-Oriented class MLP2Layer capable of handling arbitrary input, hidden, and output sizes.

Python
class MLP2Layer:
    """
    Modular 2-Layer Multi-Layer Perceptron using NumPy.
    Architecture: Input -> Dense -> Sigmoid -> Dense -> Sigmoid (Output)
    Loss Function: Binary Cross-Entropy (BCE)
    """
    def __init__(self, n_inputs, n_hidden, n_outputs, seed=42):
        rng = np.random.default_rng(seed)
        # Xavier/Glorot Initialization
        self.W1 = rng.normal(0.0, np.sqrt(2.0 / (n_inputs + n_hidden)), size=(n_inputs, n_hidden))
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = rng.normal(0.0, np.sqrt(2.0 / (n_hidden + n_outputs)), size=(n_hidden, n_outputs))
        self.b2 = np.zeros((1, n_outputs))

    def _sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    def forward(self, X):
        """Computes forward pass and stores intermediate values for backprop."""
        self.X = X
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = self._sigmoid(self.Z1)
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = self._sigmoid(self.Z2)
        return self.A2

    def backward(self, Y):
        """Computes gradients for all parameters using stored forward activations."""
        N = Y.shape[0]

        # Output layer error (BCE + Sigmoid output)
        dZ2 = (self.A2 - Y) / N
        self.dW2 = self.A1.T @ dZ2
        self.db2 = np.sum(dZ2, axis=0, keepdims=True)

        # Hidden layer error
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * self.A1 * (1.0 - self.A1)
        self.dW1 = self.X.T @ dZ1
        self.db1 = np.sum(dZ1, axis=0, keepdims=True)

    def step(self, lr):
        """Performs a single SGD parameter update step."""
        self.W1 -= lr * self.dW1
        self.b1 -= lr * self.db1
        self.W2 -= lr * self.dW2
        self.b2 -= lr * self.db2

    def fit(self, X, Y, lr=0.5, epochs=2000, verbose=False):
        """Trains the MLP over specified epochs and logs loss history."""
        loss_history = []
        for epoch in range(1, epochs + 1):
            preds = self.forward(X)
            loss = bce_loss(preds, Y)
            loss_history.append(loss)

            self.backward(Y)
            self.step(lr)

            if verbose and (epoch % (epochs // 5) == 0 or epoch == 1):
                print(f"Epoch {epoch:5d}/{epochs} | Loss: {loss:.6f}")

        return loss_history

    def predict(self, X, threshold=0.5):
        """Returns binary predictions based on the decision threshold."""
        probs = self.forward(X)
        return (probs >= threshold).astype(int)

# --- Verification of Modular MLP Class ---
print("--- Exercise 3: Testing Modular MLP Class on XOR ---")
mlp = MLP2Layer(n_inputs=2, n_hidden=4, n_outputs=1, seed=42)
history = mlp.fit(X_xor, Y_xor, lr=0.5, epochs=3000, verbose=True)

final_probs = mlp.forward(X_xor)
final_preds = mlp.predict(X_xor)

print("\nFinal Predictions Table:")
print("Input X1 X2 | Target | Probability | Predicted Class")
print("-" * 50)
for x, y, prob, pred in zip(X_xor, Y_xor, final_probs, final_preds):
    print(f"   {int(x[0])}    {int(x[1])}  |   {int(y[0])}    |   {prob[0]:.4f}    |       {pred[0]}")
Discussion & Conclusion
Gradients & Vectorization: Formulating gradients using matrix multiplications (X
T
 δ and A
1
T
​
 δ
2
​
 ) removes the need for explicit loops over sample indices, ensuring fast array operations in NumPy.

XOR Problem: The non-linear decision boundary confirms that a single hidden layer with 4 units is sufficient to learn non-linear functions.

Loss Function Selection: Binary Cross-Entropy significantly accelerates convergence compared to Mean Squared Error due to the linear term in dZ
2
​
 =
N
1
​
 (A
2
​
 −Y), eliminating output layer gradient saturation.